# BGC-Argo Data Processing

- After getting .nc files from 0.2_*_Argopy, need to process and QC the BGC-Argo data


In [1]:
# os tools
import sys
import os
import os.path
import requests
import time
import urllib3
import shutil

# data tools
import numpy                 as np
import pandas                as pd
import xarray                as xr
from   datetime              import date, datetime, timedelta                 # for saving figures with today's date
import datetime
import scipy
from   scipy.stats           import kruskal              # for boxenplot stats
from   scipy.stats           import mannwhitneyu         # for split violin plot stats

# for all plots
import matplotlib
import matplotlib.pyplot     as plt                      # needed to make map setup
import matplotlib.colors     as colors
from   matplotlib.ticker     import EngFormatter         # for degree symbol in axis
from   cmocean               import cm as cmo
import seaborn               as     sns

# for map
import shapefile
import cartopy                                           # to make map
import matplotlib.path       as     mpath                # to draw circle for map
import cartopy.crs           as     ccrs                 # for map projection
import cartopy.feature       as     cfeature             # to add land features to map
# from   xhistogram.xarray     import histogram            # for map histogram
# from   mycolorpy             import colorlist as mcp     # to get n colors list
import pyproj  
import geopandas             as     gpd                  # for adding shapefiles of frontal zones 
from   osgeo                 import gdal
# import scikit_posthocs       as     sp                   # for stats


In [2]:
from importlib import reload
import mod_plotting as crplot

from mod_plotting import setup_SO_axes
plt.rcParams.update(crplot.my_params(size=12))

In [5]:
xr.set_options(display_expand_attrs = False)

In [6]:
import shapefile
so_fronts = shapefile.Reader('./shapefiles/fronts/so_fronts.shp') 
stf_mod   = shapefile.Reader('./shapefiles/fronts/stf_mod/stf_mod.shp')

stf  = stf_mod.shape(0).points
saf  = so_fronts.shape(1).points
pf   = so_fronts.shape(2).points
sacc = so_fronts.shape(3).points
sie  = so_fronts.shape(4).points

max_latitude:          float = -30
add_gridlines:         bool  = True
color_land:            bool  = False
land_edgecolor:        str   = 'grey'
land_facecolor:        str   = 'grey'
fontsize:              float = 10
map_facecolor:         str   = 'white'
coast_linewidth:       float = 0.3
gridlines_linewidth:   float = 0.5
girdlines_color:       str   = 'grey'
gridlines_alpha:       float = 0.5
longitude_label_color: str   = 'grey'
latitude_label_color:  str   = 'grey'

In [7]:
from argopy import DataFetcher  # This is the class to work with Argo data
from argopy import ArgoIndex  #  This is the class to work with Argo index
from argopy import ArgoNVSReferenceTables  # This is the class to retrieve data from Argo reference tables
from argopy import ArgoColors  # This is a class with usefull pre-defined colors
from argopy.plot import scatter_map, scatter_plot  # This is a function to easily make maps 

# Make a fresh start
import argopy
argopy.reset_options()
argopy.clear_cache()
argopy.set_options(cachedir='cache_bgc')

In [3]:
yrtag='2018'
BOX = [-180, 180, -90, -35, 0, 2000, yrtag + '-01-01', yrtag + '-1-21']

# gdacroot = '/Volumes/ocean-repo/202501-ArgoData' 
# argopy.set_options(src='gdac', gdac = gdacroot)
argopy.set_options(src='erddap')

fetcher = DataFetcher(ds='bgc', mode='expert', params='all',
                parallel=True, progress=True,
                chunks_maxsize={'time': 30},
               )
bgc_fetcher = fetcher.region(BOX).load()


Final post-processing of the merged dataset ...


In [ ]:
# For exporting files, use today's date to avoid overwrite
datetag = str(datetime.datetime.now())[:10].replace('-','')
savepath = '/Volumes/cremas-repo/data/bgc/'

In [9]:
argopy.set_options(src='erddap')

In [9]:
# if save:
#     print('===> Saving data and index for yr ' + yrtag)
#     bgc_profiles.to_netcdf(path + 'ERDDAP_yr' + savetag + '_profiles_' + datetag + '.nc')
#     bgc_index.to_csv(path + 'ERDDAP_yr' + savetag + '_index_' + datetag + '.csv')
#     print('Successfully saved to ' + path + 'ERDDAP_yr' + savetag + '_profiles_' + datetag + '.nc')

===> Saving data and index for yr2015
Successfully saved to /Volumes/ocean-repo/CREMAS-Data/bgc/ERDDAP_yr2015_bgc_profiles_20250219.nc


In [13]:
# MAIN METHOD, bgc FLOATS: Running for multiple years 
# ~40 min per year 
save = True
savepath = '/Volumes/cremas-repo/data/bgc/'

# save = False
# yrlist = [str(x) for x in range(2018, 2024)] #2013 through 2023
yrlist=['2024']

for yrtag in yrlist:
    savetag = yrtag + '_bgc'
    # Format: [lon_min, lon_max, lat_min, lat_max, pres_min, pres_max, datim_min, datim_max]
    BOX = [-180, 180, -90, -35, 0, 2000, yrtag + '-01-01', yrtag + '-12-31']

    # Set which BGC variables are required for a float to be fetched
    # required_bgc = ['PH_IN_SITU_TOTAL']
    # fetcher = DataFetcher(ds='bgc', mode='expert', params='all',
    #                     measured = required_bgc,
    #                     parallel=True, progress=True,
    #                     chunks_maxsize={'time': 30},
                    # )
    print('===> Processing ' + savetag)
    fetcher = DataFetcher(ds='bgc', mode='expert', params='all',
                    parallel=True, progress=True,
                    chunks_maxsize={'time': 30},
                )
    bgc_fetcher = fetcher.region(BOX).load()

    # Returns xr Dataset with point observations as dimension
    bgc_profiles = bgc_fetcher.data.argo.point2profile();
    bgc_profiles = bgc_profiles.assign_attrs(raw_attrs = '', Fetched_uri='', 
                                             Fetched_constraints=str(bgc_profiles.Fetched_constraints).replace('/','_')) # simplify for save
    # Store index and domain
    bgc_index = bgc_fetcher.index
    bgc_domain = bgc_fetcher.domain 

    if save:
        print('===> Saving data and index for yr ' + yrtag)
        bgc_profiles.to_netcdf(savepath + 'ERDDAP_yr' + savetag + '_profiles_' + datetag + '.nc')
        bgc_index.to_csv(savepath + 'ERDDAP_yr' + savetag + '_index_' + datetag + '.csv')
        print('Successfully saved to ' + savepath + 'ERDDAP_yr' + savetag + '_profiles_' + datetag + '.nc')

Final post-processing of the merged dataset ...
===> Saving data and index for yr 2024
Successfully saved to /Volumes/cremas-repo/data/bgc/ERDDAP_yr2024_bgc_profiles_20250219.nc


In [18]:
# bgc_profiles = bgc_fetcher.data.argo.point2profile();
# bgc_profiles = bgc_profiles.assign_attrs(raw_attrs = '', Fetched_uri='', 
#                                              Fetched_constraints=str(bgc_profiles.Fetched_constraints).replace('/','_')) # simplify for save
# # # Store index and domain
# # bgc_index = bgc_fetcher.index
# # bgc_domain = bgc_fetcher.domain 

# save=True
# # save = False
# if save:
#     bgc_profiles.to_netcdf(path + 'ERDDAP_yr' + savetag + '_profiles_' + datetag + '.nc')
#     bgc_index.to_csv(path + 'ERDDAP_yr' + savetag + '_index_' + datetag + '.csv')
#     print('Saved data to netcdf and csv')
#     print(path + 'ERDDAP_yr' + savetag + '_profiles_' + datetag + '.nc')

Saved data to netcdf and csv
/Volumes/ocean-repo/CREMAS-Data/bgc/ERDDAP_yr2014_bgc_profiles_20250219.nc


In [ ]:
example_bgc = xr.open_dataset('/Volumes/ocean-repo/CREMAS-Data/bgc/ERDDAP_yr2014_bgc_profiles_20250219.nc')
example_bgc

<xarray.Dataset>
Dimensions:                            (N_PROF: 5384, N_LEVELS: 1415)
Coordinates:
  * N_PROF                             (N_PROF) int64 3352 2670 ... 3033 4469
  * N_LEVELS                           (N_LEVELS) int64 0 1 2 ... 1412 1413 1414
    LATITUDE                           (N_PROF) float64 ...
    LONGITUDE                          (N_PROF) float64 ...
    TIME                               (N_PROF) datetime64[ns] ...
Data variables: (12/96)
    BBP532                             (N_PROF, N_LEVELS) float32 ...
    BBP532_ADJUSTED                    (N_PROF) float32 ...
    BBP532_ADJUSTED_ERROR              (N_PROF) float32 ...
    BBP532_ADJUSTED_QC                 (N_PROF) int64 ...
    BBP532_DATA_MODE                   (N_PROF) object ...
    BBP532_QC                          (N_PROF, N_LEVELS) int64 ...
    ...                                 ...
    TEMP_ADJUSTED                      (N_PROF, N_LEVELS) float32 ...
    TEMP_ADJUSTED_ERROR                (N_PROF, N_LEVELS) float32 ...
    TEMP_ADJUSTED_QC                   (N_PROF, N_LEVELS) int64 ...
    TEMP_DATA_MODE                     (N_PROF) object ...
    TEMP_QC                            (N_PROF, N_LEVELS) int64 ...
    TIME_QC                            (N_PROF) int64 ...
Attributes:
    DATA_ID:              ARGO-BGC
    DOI:                  http://doi.org/10.17882/42182
    Fetched_from:         erddap.ifremer.fr
    Fetched_by:           sangminsong
    Fetched_date:         2025/02/19
    Fetched_constraints:  [x=-180.00_-160.00; y=-90.00_-71.67; z=0.0_500.0; t...
    Fetched_uri:          
    Processing_history:   Transformed with 'point2profile'
    raw_attrs:

## Import .nc files from 0.2_DataProcessing_Argopy

In [5]:
# Example BGC float with pH data
example_bgc = xr.open_dataset('../data/argopy_5906030_bgc_profiles_2025Jan29.nc')

In [6]:
example_bgc

<xarray.Dataset>
Dimensions:                          (N_PROF: 166, N_LEVELS: 559)
Coordinates:
  * N_PROF                           (N_PROF) int64 0 1 2 3 ... 162 163 164 165
  * N_LEVELS                         (N_LEVELS) int64 0 1 2 3 ... 556 557 558
    LATITUDE                         (N_PROF) float64 ...
    LONGITUDE                        (N_PROF) float64 ...
    TIME                             (N_PROF) datetime64[ns] ...
Data variables: (12/54)
    BBP700                           (N_PROF, N_LEVELS) float32 ...
    BBP700_ADJUSTED                  (N_PROF, N_LEVELS) float32 ...
    BBP700_ADJUSTED_ERROR            (N_PROF) float32 ...
    BBP700_ADJUSTED_QC               (N_PROF, N_LEVELS) int64 ...
    BBP700_DATA_MODE                 (N_PROF) object ...
    BBP700_QC                        (N_PROF, N_LEVELS) int64 ...
    ...                               ...
    TEMP_ADJUSTED                    (N_PROF, N_LEVELS) float32 ...
    TEMP_ADJUSTED_ERROR              (N_PROF, N_LEVELS) float32 ...
    TEMP_ADJUSTED_QC                 (N_PROF, N_LEVELS) int64 ...
    TEMP_DATA_MODE                   (N_PROF) object ...
    TEMP_QC                          (N_PROF, N_LEVELS) int64 ...
    TIME_QC                          (N_PROF) int64 ...
Attributes:
    DATA_ID:              ARGO-BGC
    DOI:                  http://doi.org/10.17882/42182
    Fetched_from:         https://erddap.ifremer.fr/erddap
    Fetched_by:           sangminsong
    Fetched_date:         2025/01/25
    Fetched_constraints:  WMO5906030
    Fetched_uri:          https://erddap.ifremer.fr/erddap/tabledap/ArgoFloat...
    history:              Transformed with point2profile

In [25]:
# temp = example_bgc.to_dataframe().reset_index()
# temp[temp.POSITION_QC == 0]

,N_PROF,N_LEVELS,BBP700,BBP700_ADJUSTED,BBP700_ADJUSTED_ERROR,BBP700_ADJUSTED_QC,BBP700_DATA_MODE,BBP700_QC,CHLA,CHLA_ADJUSTED,...,TEMP,TEMP_ADJUSTED,TEMP_ADJUSTED_ERROR,TEMP_ADJUSTED_QC,TEMP_DATA_MODE,TEMP_QC,TIME_QC,LATITUDE,LONGITUDE,TIME
0,0,0,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
1,0,1,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
2,0,2,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
3,0,3,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
5,0,5,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.634,3.634,0.002,1,D,1,1,-51.355,30.208,2019-05-01 08:58:59.000999936
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92774,165,539,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.142,3.142,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128
92775,165,540,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.139,3.139,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128
92776,165,541,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.130,3.130,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128
92777,165,542,NaN,NaN,NaN,0,A,0,NaN,NaN,...,3.125,3.125,0.002,1,D,1,1,-47.723,111.012,2023-10-13 19:34:23.002000128


In [12]:
import gsw
import mod_cremas as crx 
import mod_ocean as myocn
from mod_ocean import ytd2datetime, datetime2ytd

# Custom module
import mod_argo 
reload(mod_argo)

<module 'mod_argo' from '/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CREMAS/src/mod_argo.py'>

In [19]:
# Custom module
import mod_argo 
reload(mod_argo)

<module 'mod_argo' from '/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CREMAS/src/mod_argo.py'>

In [ ]:
# Example from SOGOS. For BGC
#  def make_float_df(wmo, floatDSdict):
#         """
#         Return dataframe from a single float dataset
#         @param:
#                 wmo (int)
#                 floatDSdict (dict): dictionary of float datasets
#         """
#         temp = floatDSdict[str(wmo)]
#         float_df = temp[['JULD','LATITUDE', 'LONGITUDE','PRES_ADJUSTED','TEMP_ADJUSTED','PSAL_ADJUSTED',
#                          'DOXY_ADJUSTED','NITRATE_ADJUSTED', # , 'PH_IN_SITU_TOTAL_ADJUSTED','BBP700_ADJUSTED', 
#                         'JULD_QC', 'POSITION_QC', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC','PSAL_ADJUSTED_QC', 
#                         'DOXY_ADJUSTED_QC', 'NITRATE_ADJUSTED_QC']].to_dataframe() #'BBP700_ADJUSTED_QC' #, 'PH_IN_SITU_TOTAL_ADJUSTED_QC',
#         dtimes = pd.to_datetime(float_df.JULD.values, unit='D', origin=pd.Timestamp('1950-01-01'))
#         float_df['yearday'] = sg.datetime2ytd(dtimes)
#         float_df['wmoid'] = np.repeat(wmo, len(float_df))


#         # need to get the profile number from the index
#         # create a 10-digit unique id so easy to sort later
#         prof = float_df.index.get_level_values(0)
#         prof = prof.astype(str); prof = [tag.zfill(3) for tag in prof]
#         float_df['profid'] = [str(wmo)+tag for tag in prof]


#         qc_keys = ['JULD_QC', 'POSITION_QC', 'PRES_ADJUSTED_QC', 'TEMP_ADJUSTED_QC', 'PSAL_ADJUSTED_QC',
#             'DOXY_ADJUSTED_QC', 'NITRATE_ADJUSTED_QC'] #, 'PH_IN_SITU_TOTAL_ADJUSTED_QC', 'BBP700_ADJUSTED_QC']
#         for key in qc_keys:  #qc flags are not stored as ints so we can convert
#                 newlist = []
#                 for qc in float_df[key]:
#                         if str(qc)[2] == 'n': newlist.append('NaN')
#                         else: newlist.append(str(qc)[2])
#                 float_df[key] = newlist
        
#         float_df.rename(columns={'LATITUDE':'lat','LONGITUDE':'lon',
#                         'PRES_ADJUSTED': 'pressure', 'TEMP_ADJUSTED': 'temperature', 'PSAL_ADJUSTED': 'salinity',
#                         'DOXY_ADJUSTED': 'oxygen', 'NITRATE_ADJUSTED':'nitrate'}, inplace=True)
#                         # 'PH_IN_SITU_TOTAL_ADJUSTED': 'pH', 'BBP700_ADJUSTED': 'bbp700'
#         float_df.rename(columns={'JULD_QC': 'juld_qc', 'POSITION_QC': 'position_qc', 
#                         'PRES_ADJUSTED_QC': 'pressure_qc', 'TEMP_ADJUSTED_QC': 'temperature_qc','PSAL_ADJUSTED_QC': 'salinity_qc',
#                         'DOXY_ADJUSTED_QC': 'oxygen_qc', 'NITRATE_ADJUSTED_QC': 'nitrate_qc'}, inplace=True) 
#                         # 'PH_IN_SITU_TOTAL_ADJUSTED_QC': 'pH_qc','BBP700_ADJUSTED_QC': 'bbp700_qc'

        
#         float_df['SA']= gsw.SA_from_SP(float_df['salinity'],float_df['pressure'],float_df['lon'],float_df['lat'])
#         float_df['CT'] = gsw.CT_from_t(float_df['SA'], float_df['temperature'], float_df['pressure']) # change Sp to SA, jun24

#         # # Add training variables Sigma0 and Spice as desired
#         float_df['sigma0'] = gsw.sigma0(float_df.SA.values, float_df.CT.values)
#         float_df['spice'] = gsw.spiciness0(float_df["SA"].values, float_df["CT"].values)

#         # Add oxygen saturation and drho/dz
#         float_df = float_df[dvars + qcvars]

#         return float_df


In [9]:
floatDF = example_bgc.to_dataframe().reset_index()

In [10]:
# # Modify to work for bgc-floats, jan 29
# # this is a modified version of mod_argo function, process_core_float()

# def process_argo_float(float_df, bgc_list = ['pH'], ref_time = '2014-01-01'):
#     """
#     Return dataframe from a single float dataset, accessed with Argopy. 
#     Assumed to be used in 'expert' mode, i.e. not quality-controlled yet for BGC.
#     (From Argopy, download and use .point2profile() and then .to_dataframe() to get input float dataframe)

#     @param:
#             float_df (dict):  single float Dataframe, accessed in 'research' mode for core
#                                 only 'expert' mode available for BGC floats
#             bgc_list : list of BGC variables to include in the final dataframe
#                         default ['pH'] adds only pH data
#             ref_time : reference time for yearday calculation
#     @return:
#             float_df (pd.DataFrame): 
#     """
#     float_df = float_df.reset_index()


#     # Default columns to rename, starting with necessary properties across core/bgc
#     # Note that Argopy "research mode" has removed "ADJUSTED" from column names
#     new_columns = {'LATITUDE':'latitude','LONGITUDE':'longitude', 'TIME':'datetime', 
#                 'CYCLE_NUMBER':'cycle_number', 'PLATFORM_NUMBER':'wmoid', 
#                 'PRES_ADJUSTED':'pressure', 'TEMP_ADJUSTED':'temperature', 'PSAL_ADJUSTED':'salinity'}
#     # Rename QC and error columns
#     new_columns.update({'TIME_QC': 'time_qc', 'POSITION_QC': 'position_qc', 
#                         'PRES_ADJUSTED_QC': 'pressure_qc', 
#                         'TEMP_ADJUSTED_QC': 'temperature_qc','PSAL_ADJUSTED_QC': 'salinity_qc'})
#     new_columns.update({'PRES_ADJUSTED_ERROR': 'pres_error', 
#                         'PSAL_ADJUSTED_ERROR': 'psal_error', 'TEMP_ADJUSTED_ERROR': 'temp_error'})
    
#     # output_vars = new_columns.values()


#     # if bgc_list == 'phys': # research mode
#     #     new_columns = {'LATITUDE':'latitude','LONGITUDE':'longitude', 'TIME':'datetime', 'CYCLE_NUMBER':'cycle_number',
#     #                             'PLATFORM_NUMBER':'wmoid', 'PRES':'pressure', 'TEMP':'temperature', 'PSAL':'salinity',
#     #                             'PRES_ERROR': 'pres_error', 'PSAL_ERROR': 'psal_error', 'TEMP_ERROR': 'temp_error'}
    
#     if 'pH' in bgc_list: # expert mode
#         new_columns.update({'PH_IN_SITU_TOTAL_ADJUSTED': 'pH', 'PH_IN_SITU_TOTAL_ADJUSTED_QC': 'pH_qc',
#                             'PH_IN_SITU_TOTAL_ADJUSTED_ERROR': 'pH_error'})
#     if 'oxygen' in bgc_list: 
#         new_columns.update({'DOXY_ADJUSTED': 'oxygen', 'DOXY_ADJUSTED_QC': 'oxygen_qc',
#                             'DOXY_ADJUSTED_ERROR': 'oxygen_error'})
#     # if 'nitrate' in bgc_list:
#     #     new_columns.update({'NITRATE_ADJUSTED': 'nitrate', 'NITRATE_ADJUSTED_QC': 'nitrate_qc',
#     #                         'NITRATE_ADJUSTED_ERROR': 'nitrate_error'})
        
#     float_df.rename(columns=new_columns, inplace=True)
#     float_df['yearday'] = myocn.datetime2ytd(float_df['datetime'], ref_time = ref_time)

#     # Create a unique profile id to be a useful index
#     # Make sure strings are filled so 1st and 10th profile are different
#     prof = [tag.zfill(3) for tag in float_df['cycle_number'].astype(str)]
#     float_df['profid'] = [str(float_df.wmoid[0]) + '_id' + tag for tag in prof]

#     # Add calculated variables using gsw
#     float_df['SA']= gsw.SA_from_SP(float_df['salinity'],float_df['pressure'],float_df['longitude'],float_df['latitude'])
#     float_df['CT'] = gsw.CT_from_t(float_df['SA'], float_df['temperature'], float_df['pressure']) 
#     float_df['sigma0'] = gsw.sigma0(float_df.SA.values, float_df.CT.values)
#     float_df['spice'] = gsw.spiciness0(float_df["SA"].values, float_df["CT"].values)

#     # Standard variable list to return (core)
#     # Can reorder by changing the output_vars list 
#     output_vars = ['wmoid', 'profid', 'latitude', 'longitude', 'datetime', 'yearday',
#             'pressure', 'CT', 'SA', 'sigma0', 'spice',
#             'temperature', 'salinity',
#             'temperature_qc', 'salinity_qc', 'pressure_qc',
#             'time_qc', 'position_qc',
#             'temp_error', 'psal_error', 'pres_error']
    
#     for x in bgc_list:
#         output_vars = output_vars + [x, x+'_qc', x+'_error']
    
#     # Turn all QC flags into strings
#     qc_vars = [var for var in float_df.columns.tolist() if '_qc' in var]
#     for k in qc_vars:
#         float_df[k] = float_df[k].astype(str)
#     # print(output_vars)
    
#     return float_df[output_vars]




In [13]:
trial = process_argo_float(floatDF, bgc_list = ['pH'], ref_time = '2014-01-01')

In [18]:
# Moved to mod_argo
#  def filter_qc_flags(float_df, qc_vars = 'all', use_flags=['1', '2', '5', '8']):
#         """
#         Filter a dataframe based on QC flags.
#         Can choose different QC flags for different variables by calling the function multiple times.
#         @param: float_df (pd.DataFrame): dataframe of float data
#                 qc_vars (list): list of QC variables to filter
#                         default 'all' filters on any variable with '_qc' in the name
#                 use_flags : flags that pass QC; default are standard argo QC flags 1, 2, and 8
#                         '1' for 'good' data (only '1' returned in 'research' mode)
#                         '2' for 'probably good' data
#                         '5' for 'changed' data (rare; for position qc where lat/lon was adjusted)
#                         '8' for 'interpolated/estimated' data
#         @return: float_qc (pd.DataFrame)
#         """ 
#         print('Using flags: ', use_flags)
#         float_qc = float_df.copy().reset_index()
#         print('# of obs before QC filtering: \t\t', len(float_qc), '\n')

#         if qc_vars == 'all':
#                 qc_vars = [var for var in float_qc.columns.tolist() if '_qc' in var]
        
#         # for var in qc_vars:
#         #         float_qc = float_qc[float_qc[var].isin(use_flags)]
#         #         print('# of obs after ', var, ': \t\t', len(float_qc))
        

#         qc_table = pd.DataFrame(columns= (use_flags + ['nobs_remaining']), index=qc_vars)
#         for var in qc_vars:
#                 for flag in use_flags:
#                         qc_table.loc[var, flag] = len(float_qc[float_qc[var] == flag])
#                 float_qc = float_qc[float_qc[var].isin(use_flags)]
#                 qc_table.loc[var, 'nobs_remaining'] = len(float_qc)
        
#         print(qc_table)
#         return float_qc
        

trial_qc_phys = filter_qc_flags(trial, qc_vars = ['temperature_qc', 'salinity_qc', 'pressure_qc', 'time_qc', 'position_qc'], use_flags=['1', '2', '5', '8'])
# trial_qc_all = filter_qc_flags(trial_qc_phys, qc_vars = ['pH_qc'], use_flags=['1', '2'])

Using flags:  ['1', '2', '5', '8']
# of obs before QC filtering: 		 92794 

                    1  2  5     8 nobs_remaining
temperature_qc  82549  0  0  9562          92111
salinity_qc     82342  0  0  9514          91856
pressure_qc     91856  0  0     0          91856
time_qc         91856  0  0     0          91856
position_qc     91856  0  0     0          91856


In [ ]:
# Make dataframes using the custom Argo module
# reload(mod_argo)
# temp = core_profiles.to_dataframe()
# core_dict = {key:mod_argo.process_core_float(group, var_list = 'default', 
#                                     ref_time= '2014-01-01') for key,group in temp.groupby('PLATFORM_NUMBER')}
